In [1]:
import os

# macOS: torch + faiss can load two OpenMP runtimes; without this, the kernel may abort.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# Conda env `tf` (Jupyter kernel): langchain-google-genai, google-generativeai, faiss-cpu, etc. are already installed.
pass

In [2]:
import os
from pathlib import Path

from langchain.document_loaders import WebBaseLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

In [3]:
def _load_dotenv_simple(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key, val = key.strip(), val.strip().strip("'\"")
        if key and key not in os.environ:
            os.environ[key] = val


for _env_path in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    _load_dotenv_simple(_env_path)


def _looks_like_real_gemini_key(k: str) -> bool:
    k = (k or "").strip()
    # Google AI Studio keys are typically ~39 chars and start with AIzaSy
    if len(k) < 35 or not k.startswith("AIza"):
        return False
    low = k.lower()
    if "enter your" in low or "placeholder" in low or "your key" in low:
        return False
    return True


api_key = (os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY") or "").strip()
USE_GEMINI = _looks_like_real_gemini_key(api_key)
if USE_GEMINI:
    os.environ["GOOGLE_API_KEY"] = api_key
else:
    print(
        "No valid GOOGLE_API_KEY/GEMINI_API_KEY — using HuggingFace embeddings (offline). "
        "Put a real key in the project `.env` to use Gemini for embeddings and answers."
    )

In [4]:
# Gemini LLM only when a real key is set (import here so offline runs avoid loading google-genai in the kernel).
if USE_GEMINI:
    from langchain_google_genai import ChatGoogleGenerativeAI

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        google_api_key=api_key,
        temperature=0.9,
        max_output_tokens=500,
    )
else:
    llm = None

### (1) Load data

In [5]:
loader = WebBaseLoader(
    [
        "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
        "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html",
    ]
)
data = loader.load()
len(data)

2

### (2) Split data to create chunks

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)

In [7]:
len(docs)

41

In [8]:
docs[0]

Document(page_content='Wall Street rises as Tesla soars on AI optimism', metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html', 'title': 'Wall Street rises as Tesla soars on AI optimism', 'description': 'Tesla (TSLA.O) rallied 10% after Morgan Stanley upgraded the electric car maker to ', 'language': 'en'})

### (3) Create embeddings for these chunks and save them to FAISS index

In [9]:
# Gemini embeddings when a key is set; otherwise local sentence-transformers (already in `tf`).
if USE_GEMINI:
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    embeddings = GoogleGenerativeAIEmbeddings(
        model="gemini-embedding-001",
        google_api_key=api_key,
    )
else:
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_index = FAISS.from_documents(docs, embeddings)

In [10]:
# Persist FAISS on disk (pickle fails: embedding clients hold thread locks).
persist_dir = "faiss_retrieval_index"
vector_index.save_local(persist_dir)

In [11]:
persist_dir = "faiss_retrieval_index"
if os.path.isdir(persist_dir):
    vector_index = FAISS.load_local(persist_dir, embeddings)

### (4) Retrieve similar embeddings for a given question and call LLM to retrieve final answer

In [12]:
query = "what is the price of Tiago iCNG?"
# query = "what are the main features of punch iCNG?"

retriever = vector_index.as_retriever()
retrieved_docs = retriever.get_relevant_documents(query)
retrieved_docs

[Document(page_content='also introduced the twin-cylinder technology on its Tiago and Tigor models.The Tiago iCNG is priced between Rs 6.55 lakh and Rs 8.1 lakh, while the Tigor iCNG comes at a price range of Rs 7.8 lakh to Rs 8.95 lakh.Tata Motors Passenger Vehicles Ltd Head-Marketing, Vinay Pant said these introductions put together will make the company\'s CNG line up "appealing, holistic, and stronger than ever".', metadata={'source': 'https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html', 'title': 'Tata Motors launches Punch iCNG, price starts at Rs 7.1 lakh', 'description': "The Punch iCNG is equipped with the company's proprietary twin-cylinder technology with enhanced safety features like a micro-switch to keep the car switched off at the time of refuelling and thermal incident protection that cuts off CNG supply to the engine and releases gas into the atmosphere, Tata Motors said in a statement.", 'language': 'en'

In [13]:
# RetrievalQAWithSourcesChain + LLMChain is incompatible with langchain-google-genai on this stack;
# call the chat model directly after retrieval.
if USE_GEMINI:
    from langchain_core.messages import HumanMessage

    context = "\n\n".join(d.page_content for d in retrieved_docs[:8])
    try:
        llm.invoke(
            [
                HumanMessage(
                    content=(
                        "Answer using only the context. Cite source URLs from the page metadata when relevant.\n\n"
                        f"Context:\n{context}\n\nQuestion: {query}"
                    )
                )
            ]
        )
    except Exception as exc:
        print(f"Gemini request failed ({type(exc).__name__}): {exc}")
else:
    print("Offline mode: retrieved chunks are in the previous cell. Add GOOGLE_API_KEY to `.env` for Gemini answers.")

Gemini request failed (ChatGoogleGenerativeAIError): Error calling model 'gemini-2.0-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 54.73571452s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about G